# SSL + Rebalance Fine-tuning with GAN-Synthetic MEL Augmentation

**Purpose:** run the existing 3-seed SSL+Rebalance fine-tuning pipeline, but with the
5x MEL oversampling (simple duplication) replaced or supplemented by the GAN-generated
synthetic MEL images from `gan-mel-synthesis.ipynb`.

**Prerequisite:** run `gan-mel-synthesis.ipynb` first. This notebook expects
`synthetic_mel_manifest.csv` in `config.CSV_DIR` and the corresponding images in
`{OUTPUT_ROOT}/synthetic_mel/`.

**Design choice, stated explicitly for the paper:** the real 35 MEL training images are
never dropped -- synthetic images are added on top, at the same effective count as the
existing 5x-oversampling condition (175 total MEL-class training instances), so this is a
like-for-like comparison against your existing "oversampling only" ablation (25.0% MEL
recall) and the full SSL+Rebalance result (41.7% MEL recall), not a different data budget.

This notebook does **not** retrain the SSL encoder -- it loads `ssl_encoder_real_24000.pth`
directly, exactly as the original fine-tuning notebook does after Stage 1 completes.


In [1]:
import os
import sys
import glob
import json
import random
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 1. Dynamically locate and import config.py
config_files = glob.glob('/kaggle/input/**/config.py', recursive=True)
if config_files:
    config_dir = os.path.dirname(config_files[0])
    if config_dir not in sys.path:
        sys.path.insert(0, config_dir)
    import config
    print(f"Loaded config from: {config_files[0]}")
else:
    raise FileNotFoundError("Could not find config.py under /kaggle/input/")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# 2. Map direct Kaggle input paths
BACKUP_DIR = getattr(config, 'CSV_DIR', '/kaggle/input/isic-2019-for-melanoma-with-csvs/CSVs-20260711T054424Z-2-001')

# Locate ISIC image directory dynamically
image_files = glob.glob('/kaggle/input/**/ISIC_*.jpg', recursive=True)
if image_files:
    LABELED_DIR = os.path.dirname(image_files[0])
else:
    LABELED_DIR = '/kaggle/input/isic-2019-for-melanoma-with-csvs/ISIC_2019_Training_Input'

print(f"Using real image directory: {LABELED_DIR}")

OUTPUT_DIR = '/kaggle/working'
SYNTH_DIR = os.path.join(OUTPUT_DIR, 'synthetic_mel')

# 3. Locate encoder weights dynamically
encoder_files = glob.glob('/kaggle/input/**/ssl_encoder_real_24000.pth', recursive=True) or glob.glob('/kaggle/input/**/ssl_encoder_real_24000.pt', recursive=True)
if not encoder_files:
    raise FileNotFoundError("Could not locate ssl_encoder_real_24000 inside /kaggle/input/")
SSL_ENCODER_PATH = encoder_files[0]
print(f"Found SSL Encoder at: {SSL_ENCODER_PATH}")

# Locate the directory containing actual synthetic image files (.jpg/.png)
synth_img_files = glob.glob('/kaggle/input/**/*.jpg', recursive=True) + glob.glob('/kaggle/input/**/*.png', recursive=True)
synth_mel_files = [f for f in synth_img_files if 'synthetic_mel_' in os.path.basename(f)]

if synth_mel_files:
    SYNTH_DIR = os.path.dirname(synth_mel_files[0])
else:
    raise FileNotFoundError("Could not find any synthetic_mel_*.jpg or *.png files inside /kaggle/input/")

print(f"Using synthetic image directory: {SYNTH_DIR}")

# Locate synthetic manifest CSV dynamically
manifest_files = glob.glob('/kaggle/input/**/synthetic_mel_manifest.csv', recursive=True)
if manifest_files:
    SYNTH_MANIFEST_PATH = manifest_files[0]
else:
    raise FileNotFoundError("Could not locate synthetic_mel_manifest.csv in /kaggle/input/")
print(f"Found Manifest at: {SYNTH_MANIFEST_PATH}")

# Verify file existence
assert os.path.exists(SSL_ENCODER_PATH), f"Encoder missing at {SSL_ENCODER_PATH}"
assert os.path.exists(SYNTH_MANIFEST_PATH), f"Manifest missing at {SYNTH_MANIFEST_PATH}"

print("All setup paths successfully verified!")

Loaded config from: /kaggle/input/datasets/ittisamurtunib/skin-ssl-code/skin-ssl-code/config.py
Device: cuda
Using real image directory: /kaggle/input/datasets/ittisamurtunib/dataset/ISIC_2019_Training_Input/ISIC_2019_Training_Input
Found SSL Encoder at: /kaggle/input/datasets/ittisamurtunib/ssl-encoder-real-24000/ssl_encoder_real_24000.pth
Using synthetic image directory: /kaggle/input/datasets/ittisamurtunib/synthetic-mel-gan-dataset/outputs/synthetic_mel
Found Manifest at: /kaggle/input/datasets/ittisamurtunib/gan-based-synthetic-generation-mel/synthetic_mel_manifest.csv
All setup paths successfully verified!


In [2]:
# =====================================================================
# LOAD SPLITS + SYNTHETIC MEL MANIFEST
# =====================================================================
# Dynamically locate CSV files in /kaggle/input/
exp_train_files = glob.glob('/kaggle/input/**/expA_train.csv', recursive=True)
exp_val_files = glob.glob('/kaggle/input/**/expA_val.csv', recursive=True)
exp_test_files = glob.glob('/kaggle/input/**/expA_test.csv', recursive=True)

if not (exp_train_files and exp_val_files and exp_test_files):
    raise FileNotFoundError("Could not locate split CSVs (expA_train.csv, expA_val.csv, expA_test.csv) in /kaggle/input/")

train_df = pd.read_csv(exp_train_files[0])
val_df = pd.read_csv(exp_val_files[0])
test_df = pd.read_csv(exp_test_files[0])

print(f"Loaded train CSV from: {exp_train_files[0]}")

synth_df = pd.read_csv(SYNTH_MANIFEST_PATH)

n_real_mel = (train_df['label'] == 'MEL').sum()
print(f"Real MEL in train split: {n_real_mel}")
print(f"Synthetic MEL available: {len(synth_df)}")

# Match the existing 5x-oversample effective count (175) for a like-for-like ablation.
# real MEL contributes its natural 1x count; synthetic images top up the rest.
TARGET_MEL_COUNT = n_real_mel * 5
n_synthetic_needed = max(0, TARGET_MEL_COUNT - n_real_mel)
synth_used = synth_df.sample(n=min(n_synthetic_needed, len(synth_df)), random_state=42).reset_index(drop=True)
print(f"Using {len(synth_used)} synthetic MEL images "
      f"(target effective MEL count = {TARGET_MEL_COUNT}, real contributes {n_real_mel})")

normal_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

mel_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(45),
    transforms.ColorJitter(0.4, 0.4, 0.3, 0.1),
    transforms.RandomAffine(15, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class SyntheticAugmentedDataset(Dataset):
    """Same behavior as the original AugmentedDataset, but MEL is supplemented with
    GAN-synthetic images (loaded from SYNTH_DIR) instead of naive 1x/5x duplication."""
    def __init__(self, df, synth_df, transform_normal, transform_mel, use_synthetic=True, seed=42):
        self.classes = sorted(df['label'].unique())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.transform_normal = transform_normal
        self.transform_mel = transform_mel

        other_rows = df[df['label'] != 'MEL'].copy()
        other_rows['source'] = 'real'
        mel_real = df[df['label'] == 'MEL'].copy()
        mel_real['source'] = 'real'

        frames = [other_rows, mel_real]
        if use_synthetic and len(synth_df) > 0:
            mel_synth = synth_df.copy()
            mel_synth['source'] = 'gan_synthetic'
            frames.append(mel_synth)

        self.expanded_df = pd.concat(frames, ignore_index=True) \
                             .sample(frac=1, random_state=seed).reset_index(drop=True)

    def __len__(self):
        return len(self.expanded_df)

    def __getitem__(self, idx):
        row = self.expanded_df.iloc[idx]
        img_dir = SYNTH_DIR if row['source'] == 'gan_synthetic' else LABELED_DIR
        
        img_name = str(row['image']).strip()
        
        # Check direct path first
        img_path = os.path.join(img_dir, img_name)
        
        # If missing extension in manifest, check standard image extensions
        if not os.path.exists(img_path):
            found = False
            for ext in ['.jpg', '.png', '.jpeg', '.JPG', '.PNG']:
                test_path = os.path.join(img_dir, f"{img_name}{ext}")
                if os.path.exists(test_path):
                    img_path = test_path
                    found = True
                    break
            if not found:
                raise FileNotFoundError(f"Image not found: {img_name} inside {img_dir}")

        img = Image.open(img_path).convert('RGB')
        label = self.class_to_idx[row['label']]
        
        if row['label'] == 'MEL' and self.transform_mel:
            img = self.transform_mel(img)
        elif self.transform_normal:
            img = self.transform_normal(img)
            
        return img, label

Loaded train CSV from: /kaggle/input/datasets/ittisamurtunib/dataset/CSVs-20260711T054424Z-2-001/CSVs/expA_train.csv
Real MEL in train split: 35
Synthetic MEL available: 200
Using 140 synthetic MEL images (target effective MEL count = 175, real contributes 35)


In [3]:
# =====================================================================
# MODEL DEFINITIONS -- identical to ssl-finetune-3seed-macrof1.ipynb
# =====================================================================

class SimpleEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten()
        )
    def forward(self, x):
        return self.features(x)

class SSLClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.encoder(x))

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.5):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return acc, macro_f1, all_labels, all_preds


In [4]:
# =====================================================================
# 3-SEED FINE-TUNING WITH SYNTHETIC-AUGMENTED MEL DATA
# Checkpoint selection by validation macro-F1, same as the original run.
# =====================================================================
weight_dict = {'MEL': 4.0, 'BKL': 2.0, 'NV': 1.0}
seeds = [42, 123, 2024]
all_results = {}

val_ds = SyntheticAugmentedDataset(val_df, pd.DataFrame(), test_transform, test_transform,
                                    use_synthetic=False)
test_ds = SyntheticAugmentedDataset(test_df, pd.DataFrame(), test_transform, test_transform,
                                     use_synthetic=False)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)

for seed in seeds:
    print(f"\n{'#'*40}\nSEED {seed}\n{'#'*40}")
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    train_ds = SyntheticAugmentedDataset(train_df, synth_used, normal_transform, mel_transform,
                                          use_synthetic=True, seed=seed)
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    print(f"Training set size (with synthetic MEL): {len(train_ds)}  "
          f"class counts: {Counter(train_ds.expanded_df['label'])}")

    encoder = SimpleEncoder()
    encoder.load_state_dict(torch.load(SSL_ENCODER_PATH, map_location=device))
    for p in encoder.parameters():
        p.requires_grad = True
    model = SSLClassifier(encoder, len(train_ds.classes)).to(device)

    weights = torch.tensor([weight_dict[c] for c in train_ds.classes],
                            dtype=torch.float32).to(device)
    criterion = FocalLoss(alpha=weights, gamma=1.5)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    epochs, patience = 30, 7
    best_val_macro_f1, epochs_no_improve, best_epoch = -1.0, 0, -1
    history = {"train_loss": [], "train_acc": [], "val_acc": [], "val_macro_f1": []}
    seed_ckpt = os.path.join(OUTPUT_DIR, f'gan_augmented_best_seed_{seed}.pth')

    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
        val_acc, val_macro_f1, _, _ = evaluate(model, val_loader)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["val_macro_f1"].append(val_macro_f1)
        print(f"Epoch {epoch+1}: Train Acc={train_acc:.3f}, Val Acc={val_acc:.3f}, "
              f"Val Macro-F1={val_macro_f1:.3f}")

        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1, best_epoch = val_macro_f1, epoch + 1
            torch.save(model.state_dict(), seed_ckpt)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(torch.load(seed_ckpt, map_location=device))
    test_acc, test_macro_f1, true_labels, pred_labels = evaluate(model, test_loader)
    report = classification_report(true_labels, pred_labels, target_names=train_ds.classes,
                                    output_dict=True)
    mel_recall = report['MEL']['recall']

    print(f"\n>>> Seed {seed}: Test Acc={test_acc:.3f}, Test Macro-F1={test_macro_f1:.3f}, "
          f"MEL Recall={mel_recall:.1%} <<<")
    print(f"Prediction distribution: {Counter(pred_labels)}")

    all_results[seed] = {
        "test_accuracy": test_acc, "test_macro_f1": test_macro_f1, "mel_recall": mel_recall,
        "best_epoch": best_epoch, "best_val_macro_f1": best_val_macro_f1,
        "report": report, "history": history,
        "n_synthetic_mel_used": len(synth_used),
    }

accs = [all_results[s]["test_accuracy"] for s in seeds]
f1s = [all_results[s]["test_macro_f1"] for s in seeds]
recalls = [all_results[s]["mel_recall"] for s in seeds]

print("\n" + "="*40)
print("3-SEED SUMMARY -- SSL + GAN-Synthetic-Augmented Rebalance")
print("="*40)
print(f"Mean Test Accuracy: {np.mean(accs):.3f} +/- {np.std(accs):.3f}")
print(f"Mean Test Macro-F1: {np.mean(f1s):.3f} +/- {np.std(f1s):.3f}")
print(f"Mean MEL Recall:    {np.mean(recalls):.1%} +/- {np.std(recalls):.1%}")
print("\nCompare against:")
print("  Oversampling only (no SSL):     25.0% +/- 10.2% MEL recall")
print("  Full SSL + Rebalance (current): 41.7% +/-  5.9% MEL recall")

with open(os.path.join(OUTPUT_DIR, 'ssl_gan_augmented_3seed_results.json'), 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"\nSaved results to ssl_gan_augmented_3seed_results.json")



########################################
SEED 42
########################################
Training set size (with synthetic MEL): 623  class counts: Counter({'NV': 349, 'MEL': 175, 'BKL': 99})
Epoch 1: Train Acc=0.284, Val Acc=0.067, Val Macro-F1=0.042
Epoch 2: Train Acc=0.329, Val Acc=0.452, Val Macro-F1=0.265
Epoch 3: Train Acc=0.526, Val Acc=0.548, Val Macro-F1=0.385
Epoch 4: Train Acc=0.615, Val Acc=0.596, Val Macro-F1=0.389
Epoch 5: Train Acc=0.612, Val Acc=0.587, Val Macro-F1=0.410
Epoch 6: Train Acc=0.644, Val Acc=0.615, Val Macro-F1=0.406
Epoch 7: Train Acc=0.657, Val Acc=0.596, Val Macro-F1=0.447
Epoch 8: Train Acc=0.644, Val Acc=0.606, Val Macro-F1=0.431
Epoch 9: Train Acc=0.684, Val Acc=0.596, Val Macro-F1=0.439
Epoch 10: Train Acc=0.701, Val Acc=0.615, Val Macro-F1=0.460
Epoch 11: Train Acc=0.698, Val Acc=0.577, Val Macro-F1=0.428
Epoch 12: Train Acc=0.666, Val Acc=0.606, Val Macro-F1=0.410
Epoch 13: Train Acc=0.706, Val Acc=0.596, Val Macro-F1=0.433
Epoch 14: Train Acc=0.

In [5]:
import glob
files = glob.glob('/kaggle/input/datasets/ittisamurtunib/gan-based-synthetic-generation-mel/**', recursive=True)
print("Sample files found:", [f for f in files if not os.path.isdir(f)][:5])

Sample files found: ['/kaggle/input/datasets/ittisamurtunib/gan-based-synthetic-generation-mel/gan_synthetic_eval.json', '/kaggle/input/datasets/ittisamurtunib/gan-based-synthetic-generation-mel/synthetic_mel_manifest.csv', '/kaggle/input/datasets/ittisamurtunib/gan-based-synthetic-generation-mel/gan_mel_generator.pth', '/kaggle/input/datasets/ittisamurtunib/gan-based-synthetic-generation-mel/synthetic_mel_grid.png']
